# DeepWiki Infrastructure Deployment Runbook

This notebook deploys DeepWiki infrastructure to Azure using ARM templates.
Each cell block can be run independently to deploy specific resources.

## Prerequisites
- Azure CLI authenticated (`az login`)
- Python 3.10 or higher with required packages installed
- Configuration file: `config.py`

## Deployment Blocks
1. **Setup & Configuration** - Load config and authenticate
2. **Resource Group** - Create Azure Resource Group
3. **Managed Identity** - Create User-Assigned Managed Identity
4. **Azure OpenAI** - Deploy Azure OpenAI Service with models
5. **Azure ML Workspace** - Deploy AML workspace with storage, Key Vault, etc.
6. **Azure Cognitive Search** - Deploy search service for code indexing

## BLOCK 1: Setup & Configuration

Load configuration from `config.py` and authenticate with Azure.

In [ ]:
# Import required libraries
import json
import importlib.util
from pathlib import Path
from azure.identity import AzureCliCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.resource.resources.models import ResourceGroup

# Import deployment utilities
from utils import deploy_arm, set_datastore_credential_to_managed_identity, provision_managed_network

print("✓ Libraries imported successfully")

In [ ]:
# Load configuration from config.py
config_path = Path("config.py")
spec = importlib.util.spec_from_file_location("config", config_path)
config_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(config_module)

# Extract configuration variables
config = {}
config_vars = [attr for attr in dir(config_module) if not attr.startswith('_')]
for var in config_vars:
    config[var] = getattr(config_module, var)

print("✓ Configuration loaded successfully")
print(f"  - Subscription ID: {config['subscription_id']}")
print(f"  - Resource Group: {config['resource_group']}")
print(f"  - Location: {config['location']}")

In [ ]:
# Authenticate with Azure using Azure CLI credentials
credential = AzureCliCredential()

# Test authentication
credential.get_token("https://management.azure.com/.default")

# Initialize resource management client
resource_client = ResourceManagementClient(credential, config['subscription_id'])

print("✓ Azure authentication successful")

## BLOCK 2: Create Resource Group

Create or verify the Azure Resource Group for DeepWiki resources.

In [ ]:
# Create Resource Group
resource_group_name = config['resource_group']
location = config['location']

print(f"Creating resource group: {resource_group_name}")
print(f"Location: {location}")

try:
    # Check if resource group exists
    existing_rg = resource_client.resource_groups.get(resource_group_name)
    print(f"✓ Resource group '{resource_group_name}' already exists")
    print(f"  - Location: {existing_rg.location}")
    print(f"  - Provisioning State: {existing_rg.properties.provisioning_state}")
except Exception:
    # Create resource group
    print(f"Creating resource group '{resource_group_name}'...")
    resource_group_params = ResourceGroup(location=location)
    result = resource_client.resource_groups.create_or_update(
        resource_group_name,
        resource_group_params
    )
    print(f"✓ Resource group created successfully")
    print(f"  - Location: {result.location}")
    print(f"  - Provisioning State: {result.properties.provisioning_state}")

## BLOCK 3: Deploy Azure OpenAI Service

Deploy Azure OpenAI service with model deployments using ARM template.

**Note:** Run this block only if `is_creating_open_ai_endpoint = True` in config.py

In [ ]:
# Deploy Azure OpenAI
if config['is_creating_open_ai_endpoint']:
    print("=" * 60)
    print("DEPLOYING: Azure OpenAI Service")
    print("=" * 60)
    print(f"Resource Name: {config['open_ai_resource_name']}")
    print(f"Location: {config['location']}")
    print(f"Main Model: {config['open_ai_main_model_model_name']}")
    print(f"Embedding Model: {config['open_ai_embedding_model_name']}")
    
    # Load ARM template
    template_path = Path("templates/AOAI.Template.json")
    with open(template_path, 'r') as f:
        template = json.load(f)
    
    # Build parameters from config
    parameters = {
        "location": {"value": config['location']},
        "openAiResourceName": {"value": config['open_ai_resource_name']},
        "openAiMainModelDeploymentName": {"value": config['open_ai_main_model_deployment_name']},
        "openAiMainModelModelName": {"value": config['open_ai_main_model_model_name']},
        "openAiMainModelTokensPerMinute": {"value": config['open_ai_main_model_tokens_per_minute']},
        "openAiMainModelVersion": {"value": config['open_ai_main_model_version']},
        "openAiMainModelSkuName": {"value": config['open_ai_main_model_sku_name']},
        "isDeployingSecondaryAgentModel": {"value": config['is_deploying_secondary_agent_model']},
        "openAiAgentModelDeploymentName": {"value": config['open_ai_agent_model_deployment_name']},
        "openAiAgentModelModelName": {"value": config['open_ai_agent_model_model_name']},
        "openAiAgentModelTokensPerMinute": {"value": config['open_ai_agent_model_tokens_per_minute']},
        "openAiAgentModelVersion": {"value": config['open_ai_agent_model_version']},
        "openAiAgentModelSkuName": {"value": config['open_ai_agent_model_sku_name']},
        "isCreatingOpenAiEndpoint": {"value": config['is_creating_open_ai_endpoint']},
        "isDeployingReasoningModel": {"value": config['is_deploying_reasoning_model']},
        "openAiReasoningModelDeploymentName": {"value": config['open_ai_reasoning_model_deployment_name']},
        "openAiReasoningModelModelName": {"value": config['open_ai_reasoning_model_model_name']},
        "openAiReasoningModelTokensPerMinute": {"value": config['open_ai_reasoning_model_tokens_per_minute']},
        "openAiReasoningModelVersion": {"value": config['open_ai_reasoning_model_version']},
        "openAiReasoningModelSkuName": {"value": config['open_ai_reasoning_model_sku_name']},
        "isDeployingIndexingModel": {"value": config['is_deploying_indexing_model']},
        "openAiIndexingModelDeploymentName": {"value": config['open_ai_indexing_model_deployment_name']},
        "openAiIndexingModelModelName": {"value": config['open_ai_indexing_model_model_name']},
        "openAiIndexingModelTokensPerMinute": {"value": config['open_ai_indexing_model_tokens_per_minute']},
        "openAiIndexingModelVersion": {"value": config['open_ai_indexing_model_version']},
        "openAiIndexingModelSkuName": {"value": config['open_ai_indexing_model_sku_name']},
        "openAiEmbeddingDeploymentName": {"value": config['open_ai_embedding_deployment_name']},
        "openAiEmbeddingModelName": {"value": config['open_ai_embedding_model_name']},
        "openAiEmbeddingTokensPerMinute": {"value": config['open_ai_embedding_tokens_per_minute']},
        "openAiEmbeddingVersion": {"value": config['open_ai_embedding_version']},
        "openAiEmbeddingSkuName": {"value": config['open_ai_embedding_sku_name']},
        "resourceTags": {"value": {}}
    }
    
    # Create deployment properties
    deployment_properties = DeploymentProperties(
        mode=DeploymentMode.incremental,
        template=template,
        parameters=parameters
    )
    
    # Start deployment
    deployment_name = f"deepwiki-aoai-{config['location']}"
    print(f"\nStarting deployment: {deployment_name}")
    print("This may take 5-10 minutes...")
    
    try:
        deployment_async_operation = resource_client.deployments.begin_create_or_update(
            config['resource_group'],
            deployment_name,
            Deployment(properties=deployment_properties)
        )
        
        # Wait for completion
        deployment_result = deployment_async_operation.result()
        
        print(f"\n✓ Azure OpenAI deployment completed successfully")
        print(f"  - Provisioning State: {deployment_result.properties.provisioning_state}")
        print(f"  - Resource Name: {config['open_ai_resource_name']}")
        
    except Exception as e:
        print(f"\n✗ Azure OpenAI deployment failed: {e}")
        raise
else:
    print("=" * 60)
    print("SKIPPING: Azure OpenAI Service")
    print("=" * 60)
    print(f"Reason: is_creating_open_ai_endpoint = False in config.py")
    print(f"Using existing Azure OpenAI resource: {config['open_ai_resource_name']}")

## BLOCK 4: Deploy Azure Machine Learning Workspace

Deploy Azure ML workspace with all dependent resources (Storage, Key Vault, Container Registry, Application Insights) using ARM template.

In [ ]:
# Deploy Azure Machine Learning Workspace
print("=" * 60)
print("DEPLOYING: Azure Machine Learning Workspace")
print("=" * 60)
print(f"Workspace Name: {config['machine_learning_workspace_name']}")
print(f"Location: {config['location']}")
print(f"Sub Components Prefix: {config['machine_learning_workspace_sub_components_name_prefix']}")

# Build parameters from config
parameters = {
    "location": config['location'],
    "tenantId": config['tenant_id'],
    "machineLearningWorkspaceName": config['machine_learning_workspace_name'],
    "machineLearningWorkspaceSubComponentsNamePrefix": config['machine_learning_workspace_sub_components_name_prefix'],
    "deepWikiIdentityName": config['identity_name'],
    "openAiResourceName": config['open_ai_resource_name'],
    "deploymentIdentityPrincipalId": config['deployment_identity_principal_id'],
    "deploymentIdentityPrincipalType": "User",
    "isOpenAiResourceManagedByDeepWiki": config['is_creating_open_ai_endpoint'],
    "isCreatingVnetForAzureML": config['is_creating_vnet_for_azure_ml'],
    "isEnablingDiskEncryptionForAzureML": config['is_enabling_disk_encryption_for_azure_ml'],
    "resourceTags": {},
    "emailAliases": []
}

# Deploy using utility function
deployment_name = f"deepwiki-aml-{config['location']}"
print(f"\nStarting deployment: {deployment_name}")
print("This may take 10-15 minutes...")
print("Creating:")
print("  - User-Assigned Managed Identity")
print("  - Storage Account (ADLS Gen2)")
print("  - Key Vault")
print("  - Container Registry")
print("  - Application Insights")
print("  - Log Analytics Workspace")
print("  - Azure Machine Learning Workspace")

try:
    outputs = deploy_arm(
        template_file_path="templates/AML.Template.json",
        deployment_name=deployment_name,
        parameters=parameters,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group'],
        skip_role_assignment=False
    )
    
    print(f"\n✓ Azure ML deployment completed successfully")
    print(f"  - Workspace Name: {config['machine_learning_workspace_name']}")
    print(f"  - Managed Identity: {config['identity_name']}")
    
except Exception as e:
    print(f"\n✗ Azure ML deployment failed: {e}")
    raise


## BLOCK 4a: Update AML Datastore to Use Managed Identity

Update the default datastore to use managed identity authentication instead of account key for enhanced security.

In [ ]:
# Update default datastores to use managed identity authentication
try:
    set_datastore_credential_to_managed_identity(
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group'],
        workspace_name=config['machine_learning_workspace_name']
    )
except Exception as e:
    print(f"\n⚠ Note: Datastore update encountered issues: {e}")
    print("You can continue with the deployment and update datastores manually later if needed.")

## BLOCK 4b: Provision Azure ML Managed Network

After workspace creation, the managed network needs to be explicitly provisioned to activate the outbound rules.
This step is required when `is_creating_vnet_for_azure_ml = True`.

**Note:** The workspace managed identity requires the "Azure AI Enterprise Network Connection Approver" role on the storage account. This role assignment is included in the ARM template, but Azure may need a few minutes to propagate the permissions. If you encounter a permissions error, wait 2-3 minutes and retry this cell.

In [ ]:
# Provision Managed Network for Azure ML Workspace
if config['is_creating_vnet_for_azure_ml']:
    try:
        provision_managed_network(
            subscription_id=config['subscription_id'],
            resource_group=config['resource_group'],
            workspace_name=config['machine_learning_workspace_name'],
            include_spark=False
        )
    except Exception as e:
        print(f"\n⚠ Note: Managed network provisioning encountered issues: {e}")
        print("You can manually provision it from Azure Portal if needed.")
else:
    print("=" * 60)
    print("SKIPPING: Azure ML Managed Network Provisioning")
    print("=" * 60)
    print("Reason: is_creating_vnet_for_azure_ml = False in config.py")

## BLOCK 5: Deploy Azure Cognitive Search

Deploy Azure Cognitive Search service for code indexing and semantic search using ARM template.

In [ ]:
# Deploy Azure Cognitive Search
print("=" * 60)
print("DEPLOYING: Azure Cognitive Search Service")
print("=" * 60)
print(f"Search Service Name: {config['search_service_name']}")
print(f"Location: {config['location']}")

# Build parameters from config
parameters = {
    "location": config['location'],
    "searchServiceName": config['search_service_name'],
    "machineLearningWorkspaceSubComponentsNamePrefix": config['machine_learning_workspace_sub_components_name_prefix'],
    "deepWikiIdentityName": config['identity_name'],
    "deploymentIdentityPrincipalId": config['deployment_identity_principal_id'],
    "deploymentIdentityPrincipalType": "User",
    "resourceTags": {}
}

# Deploy using utility function
deployment_name = f"deepwiki-acs-{config['location']}"
print(f"\nStarting deployment: {deployment_name}")
print("This may take 5-10 minutes...")
print("Creating:")
print("  - Azure Cognitive Search Service")
print("  - Role assignments for managed identity")

try:
    outputs = deploy_arm(
        template_file_path="templates/ACS.Template.json",
        deployment_name=deployment_name,
        parameters=parameters,
        subscription_id=config['subscription_id'],
        resource_group=config['resource_group'],
        skip_role_assignment=False
    )
    
    print(f"\n✓ Azure Cognitive Search deployment completed successfully")
    print(f"  - Search Service Name: {config['search_service_name']}")
    print(f"  - Endpoint: https://{config['search_service_name']}.search.windows.net")
    
except Exception as e:
    print(f"\n✗ Azure Cognitive Search deployment failed: {e}")
    raise


## BLOCK 5a: Configure Azure Cognitive Search Data Source

Configure the Azure Cognitive Search data source to connect to the Azure ML storage account blob container.

In [ ]:
# Configure Azure Cognitive Search Data Source
from azure.search.documents.indexes import SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndexerDataSourceConnection,
    SearchIndexerDataContainer
)
from azure.ai.ml import MLClient

print("=" * 60)
print("CONFIGURING: Azure Cognitive Search Data Source")
print("=" * 60)

# Get Azure ML workspace to retrieve storage account and container details
ml_client = MLClient(
    credential=credential,
    subscription_id=config['subscription_id'],
    resource_group_name=config['resource_group'],
    workspace_name=config['machine_learning_workspace_name']
)

# Get the default blob datastore to get the container name
datastore = ml_client.datastores.get("workspaceblobstore")
storage_account_name = datastore.account_name
container_name = datastore.container_name

print(f"Storage Account: {storage_account_name}")
print(f"Container: {container_name}")
print(f"Search Service: {config['search_service_name']}")

# Create connection string for managed identity authentication
# Format: ResourceId=/subscriptions/{subscription}/resourceGroups/{rg}/providers/Microsoft.Storage/storageAccounts/{account};
connection_string = f"ResourceId=/subscriptions/{config['subscription_id']}/resourceGroups/{config['resource_group']}/providers/Microsoft.Storage/storageAccounts/{storage_account_name};"

# Create SearchIndexerClient using Azure CLI credential (Search service has disableLocalAuth=true)
search_endpoint = f"https://{config['search_service_name']}.search.windows.net"
indexer_client = SearchIndexerClient(
    endpoint=search_endpoint,
    credential=credential
)

# Create data source connection
data_source_name = "deepwiki-storage-datasource"
container = SearchIndexerDataContainer(name=container_name)

data_source = SearchIndexerDataSourceConnection(
    name=data_source_name,
    type="azureblob",
    connection_string=connection_string,
    container=container,
    description="DeepWiki storage account data source for code indexing"
)

try:
    # Create or update the data source
    result = indexer_client.create_or_update_data_source_connection(data_source)
    print(f"\n✓ Data source configured successfully")
    print(f"  - Data Source Name: {data_source_name}")
    print(f"  - Type: Azure Blob Storage")
    print(f"  - Container: {container_name}")
    print(f"  - Authentication: System-Assigned Managed Identity")
    print(f"\nNote: The Search service will use its system-assigned managed identity to access the storage account.")
    print(f"      This data source can be used when creating search indexes for code repositories.")
    
except Exception as e:
    print(f"\n✗ Data source configuration failed: {e}")
    raise


## BLOCK 5b: Create Azure Cognitive Search Index

Create the code search index with vector search capabilities for code repository indexing.

In [ ]:
# Create Azure Cognitive Search Index
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SimpleField,
    SearchableField,
    SearchField,
    SearchFieldDataType,
    VectorSearch,
    HnswAlgorithmConfiguration,
    VectorSearchProfile,
    SemanticConfiguration,
    SemanticSearch,
    SemanticPrioritizedFields,
    SemanticField,
    FieldMapping
)
import json
from pathlib import Path

print("=" * 60)
print("CREATING: Azure Cognitive Search Index")
print("=" * 60)

# Load index schema
schema_path = Path("index/code_index_schema.json")
with open(schema_path, 'r') as f:
    index_schema = json.load(f)

index_name = index_schema['index_name']
print(f"Index Name: {index_name}")

# Create SearchIndexClient using Azure CLI credential
search_endpoint = f"https://{config['search_service_name']}.search.windows.net"
index_client = SearchIndexClient(
    endpoint=search_endpoint,
    credential=credential
)

# Create fields from schema
fields = []
for field_def in index_schema['fields']:
    field_type = field_def.get('field_type', 'SearchField')
    
    if field_type == 'SimpleField':
        field = SimpleField(
            name=field_def['name'],
            type=field_def['type'],
            key=field_def.get('key', False),
            filterable=field_def.get('filterable', False),
            sortable=field_def.get('sortable', False),
            facetable=field_def.get('facetable', False)
        )
    elif field_type == 'SearchableField':
        field = SearchableField(
            name=field_def['name'],
            type=field_def['type'],
            searchable=field_def.get('searchable', True),
            filterable=field_def.get('filterable', False),
            sortable=field_def.get('sortable', False),
            facetable=field_def.get('facetable', False)
        )
    else:  # Vector field
        if 'vector_search_dimensions' in field_def:
            field = SearchField(
                name=field_def['name'],
                type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
                searchable=True,
                vector_search_dimensions=field_def['vector_search_dimensions'],
                vector_search_profile_name=field_def['vector_search_configuration']
            )
        else:
            field = SearchField(
                name=field_def['name'],
                type=field_def['type'],
                searchable=field_def.get('searchable', True),
                filterable=field_def.get('filterable', False),
                sortable=field_def.get('sortable', False),
                facetable=field_def.get('facetable', False)
            )
    
    fields.append(field)

# Configure vector search with HNSW algorithm
vector_search = VectorSearch(
    algorithms=[
        HnswAlgorithmConfiguration(
            name="myHnsw",
            parameters={
                "m": 4,
                "efConstruction": 400,
                "efSearch": 500,
                "metric": "cosine"
            }
        )
    ],
    profiles=[
        VectorSearchProfile(
            name="myHnswProfile",
            algorithm_configuration_name="myHnsw"
        )
    ]
)

# Configure semantic search
prioritized_fields = index_schema['prioritized_fields']
semantic_config = SemanticConfiguration(
    name=index_schema['semanticConfiguration'],
    prioritized_fields=SemanticPrioritizedFields(
        title_field=SemanticField(field_name=prioritized_fields['title_field']),
        content_fields=[SemanticField(field_name=prioritized_fields['prioritized_content_fields'])]
    )
)
semantic_search = SemanticSearch(configurations=[semantic_config])

# Create the search index
index = SearchIndex(
    name=index_name,
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search
)

try:
    result = index_client.create_or_update_index(index)
    print(f"\n✓ Search index created successfully")
    print(f"  - Index Name: {result.name}")
    print(f"  - Fields: {len(result.fields)}")
    print(f"  - Vector Search: Enabled (HNSW)")
    print(f"  - Semantic Search: Enabled ({index_schema['semanticConfiguration']})")
    print(f"\nNote: The index is ready to receive documents for code search.")
    
except Exception as e:
    print(f"\n✗ Index creation failed: {e}")
    raise

## BLOCK 5c: Create Azure Cognitive Search Indexer

Create the search indexer to automatically crawl the data source and populate the index with code documents.

In [ ]:
# Create Azure Cognitive Search Indexer
from azure.search.documents.indexes.models import (
    SearchIndexer,
    IndexingSchedule
)

print("=" * 60)
print("CREATING: Azure Cognitive Search Indexer")
print("=" * 60)

# Get the data source and index names
data_source_name = "deepwiki-storage-datasource"
indexer_name = f"{index_name}-indexer"

print(f"Indexer Name: {indexer_name}")
print(f"Data Source: {data_source_name}")
print(f"Target Index: {index_name}")

# Create field mappings from schema
field_mappings = []
for mapping in index_schema.get('field_mappings', []):
    field_mappings.append(
        FieldMapping(
            source_field_name=mapping['source_field_name'],
            target_field_name=mapping['target_field_name']
        )
    )

# Create indexing schedule (runs every 24 hours - maximum allowed interval for ACS)
schedule_interval = "PT24H"  # ISO 8601 duration format (24 hours)
schedule = IndexingSchedule(interval=schedule_interval)

# Create the indexer
indexer = SearchIndexer(
    name=indexer_name,
    data_source_name=data_source_name,
    target_index_name=index_name,
    description="Indexer for DeepWiki code repository documents",
    schedule=schedule,
    field_mappings=field_mappings if field_mappings else None,
    parameters={
        "batchSize": 100,
        "maxFailedItems": 10,
        "maxFailedItemsPerBatch": 5,
        "configuration": {
            "parsingMode": "json",
            "indexedFileNameExtensions": ".json"
        }
    }
)

try:
    # Create or update the indexer
    result = indexer_client.create_or_update_indexer(indexer)
    print(f"\n✓ Search indexer created successfully")
    print(f"  - Indexer Name: {result.name}")
    print(f"  - Schedule: Every {schedule_interval} (24 hours - maximum allowed)")
    print(f"  - Status: {result.status if hasattr(result, 'status') else 'Created'}")
    
    # Run the indexer immediately
    print(f"\nRunning indexer for the first time...")
    indexer_client.run_indexer(indexer_name)
    print(f"✓ Indexer started")
    print(f"\nNote: The indexer is now crawling the data source.")
    print(f"      You can check its status in Azure Portal or by querying the indexer status.")
    print(f"      Future runs will occur automatically every {schedule_interval}.")
    
except Exception as e:
    print(f"\n✗ Indexer creation failed: {e}")
    raise

## Deployment Summary

Check the status of all deployments.

In [ ]:
# Print deployment summary
print("=" * 60)
print("DEPLOYMENT SUMMARY")
print("=" * 60)
print(f"\n✓ BLOCK 1: Setup & Configuration")
print(f"  - Configuration loaded from config.py")
print(f"  - Azure CLI authentication successful")
print(f"  - Subscription: {config['subscription_id']}")
print(f"  - Resource Group: {config['resource_group']}")
print(f"  - Location: {config['location']}")

# Check Block 2 status
try:
    rg = resource_client.resource_groups.get(config['resource_group'])
    print(f"\n✓ BLOCK 2: Resource Group")
    print(f"  - Name: {config['resource_group']}")
    print(f"  - Location: {rg.location}")
    print(f"  - Status: {rg.properties.provisioning_state}")
except:
    print(f"\n⚠ BLOCK 2: Resource Group - Not verified")

# Check Block 3 status (Azure OpenAI)
if config['is_creating_open_ai_endpoint']:
    print(f"\n⚠ BLOCK 3: Azure OpenAI Service - Ready to deploy")
    print(f"  - Resource Name: {config['open_ai_resource_name']}")
    print(f"  - Run Cell 9 to deploy")
else:
    print(f"\n➜ BLOCK 3: Azure OpenAI Service - Using existing resource")
    print(f"  - Resource Name: {config['open_ai_resource_name']}")

# Check Block 4 status (Azure ML)
print(f"\n⚠ BLOCK 4: Azure ML Workspace - Ready to deploy")
print(f"  - Workspace Name: {config['machine_learning_workspace_name']}")
print(f"  - Run Cell 11 to deploy")

# Check Block 5 status (Azure Cognitive Search)
from azure.mgmt.search import SearchManagementClient
search_mgmt = SearchManagementClient(credential, config['subscription_id'])
try:
    search_svc = search_mgmt.services.get(config['resource_group'], config['search_service_name'])
    print(f"\n✓ BLOCK 5: Azure Cognitive Search")
    print(f"  - Service Name: {config['search_service_name']}")
    print(f"  - Status: {search_svc.provisioning_state}")
    print(f"  - Endpoint: https://{config['search_service_name']}.search.windows.net")
    
    # Check Block 5a status (Data Source)
    if 'result' in dir() and hasattr(result, 'name'):
        print(f"\n✓ BLOCK 5a: Search Data Source Configuration")
        print(f"  - Data Source: deepwiki-storage-datasource")
        print(f"  - Storage Account: {storage_account_name}")
        print(f"  - Container: {container_name}")
        print(f"  - Authentication: System-Assigned Managed Identity")
    else:
        print(f"\n⚠ BLOCK 5a: Search Data Source - Ready to configure")
        print(f"  - Run Cell 19 to configure data source")
except:
    print(f"\n⚠ BLOCK 5: Azure Cognitive Search - Ready to deploy")
    print(f"  - Service Name: {config['search_service_name']}")
    print(f"  - Run Cell 17 to deploy")

print("\n" + "=" * 60)
print("NEXT STEPS:")
print("=" * 60)
if config['is_creating_open_ai_endpoint']:
    print("1. Run Cell 9 (Block 3) to deploy Azure OpenAI")
print("2. Run Cell 11 (Block 4) to deploy Azure ML Workspace")
print("3. Run Cell 13 (Block 4a) to update datastores to managed identity")
print("4. Run Cell 15 (Block 4b) to provision managed network")
print("=" * 60)